# Laboratorio 1: SQL para Científicos de Datos

Este laboratorio desarrolla competencias en extracción, filtrado, agregación y combinación de datos mediante SQL aplicado a bases de datos relacionales. Los ejercicios utilizan un dataset realista de una plataforma de streaming musical, permitiendo practicar consultas SQL en contextos similares a los encontrados en entornos profesionales.

## Objetivos del laboratorio

- Construir consultas SELECT para extraer datos específicos de bases de datos relacionales
- Aplicar filtros con WHERE para aislar subconjuntos relevantes de datos
- Calcular métricas agregadas con funciones de agregación y GROUP BY
- Combinar información de múltiples tablas usando diferentes tipos de JOIN
- Interpretar resultados de consultas para generar insights de negocio
- Escribir SQL legible y mantenible siguiendo buenas prácticas

## Dataset: Plataforma de Streaming Musical

El laboratorio utiliza una base de datos SQLite con las siguientes tablas:

- **artistas** (~50 registros): artista_id, nombre_artista, genero, pais_origen, seguidores  
  *Artistas reales de Spotify — [Spotify Tracks Dataset, Kaggle](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)*
- **canciones** (~560 registros): cancion_id, artista_id, titulo, duracion_segundos, popularidad, reproducciones_totales  
  *Canciones reales de Spotify (titulo, duración y popularidad); reproducciones_totales extrapolado desde el score de popularidad*
- **usuarios** (500 registros): usuario_id, nombre_usuario, pais, ciudad, tipo_suscripcion, fecha_registro, edad
- **reproducciones** (15,000 registros): reproduccion_id, usuario_id, cancion_id, fecha_reproduccion, completo
- **playlists** (200 registros): playlist_id, usuario_id, nombre_playlist, fecha_creacion, publica
- **playlist_canciones** (~6,200 registros): id, playlist_id, cancion_id, posicion, fecha_agregada


## Parte 1: Configuración y Exploración Inicial

Conexión a la base de datos y exploración de su estructura.

In [1]:
# Importar librerías necesarias
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("Set2")
plt.rcParams["figure.figsize"] = (10, 6)

print("✓ Librerías importadas correctamente")

✓ Librerías importadas correctamente


In [2]:
# La base de datos está incluida en el repositorio del curso.
# No se requiere descarga externa ni configuración adicional.
DB_PATH = "data/streaming_musical.db"

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(
        f"No se encontró la base de datos en '{DB_PATH}'. "
        "Verifica que el notebook se ejecuta desde el directorio 'modulo1-sql/'."
    )

# Conexión a la base de datos SQLite
conn = sqlite3.connect(DB_PATH)

# Funcion para correr los queries desde el notebook y obtener resultados como DataFrame
def ejecutar_query(query, mostrar_shape=True):
    """
    Ejecuta una consulta SQL sobre la base de datos y retorna el resultado como DataFrame.

    Parametros:
        query (str): Consulta SQL a ejecutar.
        mostrar_shape (bool): Si es True, imprime el numero de filas y columnas del resultado.

    Retorna:
        pd.DataFrame: Resultado de la consulta.
    """
    df = pd.read_sql_query(query, conn)
    if mostrar_shape:
        print(f"Resultado: {df.shape[0]} filas x {df.shape[1]} columnas")
    return df

print(f"Conectado a: {DB_PATH}")


Conectado a: data/streaming_musical.db


### Ejercicio 1.1: Listar tablas disponibles

Explorar qué tablas existen en la base de datos usando la tabla de metadatos `sqlite_master`.

In [3]:
# Listar todas las tablas
query = """
SELECT *
FROM sqlite_master
"""

tablas = ejecutar_query(query, mostrar_shape=False)
print("\nTablas disponibles:")
print(tablas)


Tablas disponibles:
    type                name            tbl_name  rootpage  \
0  table            artistas            artistas        10   
1  table           canciones           canciones        11   
2  table            usuarios            usuarios         2   
3  table      reproducciones      reproducciones         9   
4  table           playlists           playlists        20   
5  table  playlist_canciones  playlist_canciones       152   

                                                 sql  
0  CREATE TABLE "artistas" (\n"artista_id" INTEGE...  
1  CREATE TABLE "canciones" (\n"cancion_id" INTEG...  
2  CREATE TABLE "usuarios" (\n"usuario_id" INTEGE...  
3  CREATE TABLE "reproducciones" (\n"reproduccion...  
4  CREATE TABLE "playlists" (\n"playlist_id" INTE...  
5  CREATE TABLE "playlist_canciones" (\n"id" INTE...  


In [4]:
# Listar todas las tablas
query = """
SELECT name, type
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

tablas = ejecutar_query(query, mostrar_shape=False)
print("\nTablas disponibles:")
print(tablas)


Tablas disponibles:
                 name   type
0            artistas  table
1           canciones  table
2  playlist_canciones  table
3           playlists  table
4      reproducciones  table
5            usuarios  table


### Ejercicio 1.2: Explorar estructura de tablas

Examinar las columnas, tipos de datos y restricciones de cada tabla.

In [5]:
# Ver estructura de la tabla usuarios
query_info = "PRAGMA table_info(usuarios);"
info = ejecutar_query(query_info, mostrar_shape=False)
print("Columnas en 'usuarios':")
print(info[["name", "type"]])

Columnas en 'usuarios':
               name     type
0        usuario_id  INTEGER
1    nombre_usuario     TEXT
2              pais     TEXT
3            ciudad     TEXT
4  tipo_suscripcion     TEXT
5    fecha_registro     TEXT
6              edad     REAL


### Ejercicio 1.3: Vista previa de datos

Extraer las primeras filas de cada tabla para entender su contenido.

In [6]:
# SELECT básico: primeros 5 usuarios
query = """
SELECT *
FROM usuarios
LIMIT 5;
"""

usuarios_muestra = ejecutar_query(query)
usuarios_muestra

Resultado: 5 filas x 7 columnas


,usuario_id,nombre_usuario,pais,ciudad,tipo_suscripcion,fecha_registro,edad
0,1,user_0001,Colombia,Cali,Family,2024-02-20,20.0
1,2,user_0002,España,Valencia,Premium,2025-09-19,40.0
2,3,user_0003,Argentina,Rosario,Premium,2025-04-02,58.0
3,4,user_0004,Argentina,Buenos Aires,Family,2025-03-16,26.0
4,5,user_0005,Colombia,Cartagena,Student,2024-12-07,65.0


In [7]:
# Primeros 5 artistas
query = """
SELECT *
FROM artistas
LIMIT 5;
"""

artistas_muestra = ejecutar_query(query)
artistas_muestra

Resultado: 5 filas x 5 columnas


,artista_id,nombre_artista,genero,pais_origen,seguidores
0,1,Olivia Rodrigo,Pop,Estados Unidos,42895953
1,2,One Direction,Pop,Reino Unido,42210331
2,3,Lil Nas X,Hip Hop,Estados Unidos,35654573
3,4,Eminem,Hip Hop,Estados Unidos,30576418
4,5,Måneskin,Indie,Estados Unidos,24571296


### Ejercicio 1.4: Selección de columnas específicas

En lugar de usar `SELECT *`, especificar solo las columnas necesarias mejora el rendimiento y la legibilidad.

In [ ]:
# Seleccionar solo columnas relevantes de canciones
query = """
SELECT
    cancion_id,
    titulo,
    duracion_segundos,
    popularidad
FROM canciones
LIMIT 10;
"""

canciones_info = ejecutar_query(query)
canciones_info


---

## Parte 2: Filtrado y Ordenamiento

Uso de WHERE para filtrar datos y ORDER BY para ordenar resultados.

### Ejercicio 2.1: Filtrado simple con WHERE

Identificar usuarios con suscripción Premium para un análisis de retención.

In [ ]:
# Usuarios Premium
query = """
SELECT
    usuario_id,
    nombre_usuario,
    pais,
    tipo_suscripcion,
    fecha_registro
FROM usuarios
WHERE tipo_suscripcion = 'Premium'
LIMIT 10;
"""

usuarios_premium = ejecutar_query(query)
usuarios_premium

### Ejercicio 2.2: Operadores de comparación

Filtrar canciones de alta popularidad (score > 75) para curar playlists editoriales.

In [ ]:
# Canciones de alta popularidad para playlists editoriales
query = """
SELECT
    titulo,
    popularidad,
    duracion_segundos,
    reproducciones_totales
FROM canciones
WHERE popularidad > 75
ORDER BY popularidad DESC
LIMIT 15;
"""

canciones_populares = ejecutar_query(query)
canciones_populares

### Ejercicio 2.3: Operadores lógicos (AND, OR)

Identificar usuarios Premium de España o México para campaña regional.

In [ ]:
# Usuarios Premium de España o México
query = """
SELECT
    usuario_id,
    nombre_usuario,
    pais,
    ciudad,
    tipo_suscripcion
FROM usuarios
WHERE tipo_suscripcion = 'Premium'
  AND (pais = 'España' OR pais = 'México')
LIMIT 20;
"""

usuarios_target = ejecutar_query(query)
usuarios_target

### Ejercicio 2.4: Operador IN para múltiples valores

Filtrar artistas de géneros específicos de interés para curación de contenido.

In [ ]:
# Artistas de Pop, Rock o Hip Hop
query = """
SELECT
    nombre_artista,
    genero,
    seguidores
FROM artistas
WHERE genero IN ('Pop', 'Rock', 'Hip Hop')
ORDER BY seguidores DESC;
"""

artistas_principales = ejecutar_query(query)
artistas_principales.head(15)

### Ejercicio 2.5: LIKE para búsqueda por patrones

Buscar playlists relacionadas con "workout" o "gym" para análisis de fitness.

In [ ]:
# Playlists de workout
query = """
SELECT
    playlist_id,
    nombre_playlist,
    fecha_creacion,
    publica
FROM playlists
WHERE nombre_playlist LIKE '%Workout%'
   OR nombre_playlist LIKE '%workout%'
ORDER BY fecha_creacion DESC;
"""

playlists_workout = ejecutar_query(query)
playlists_workout

### Ejercicio 2.6: Manejo de NULL

Identificar artistas sin país de origen registrado para completar metadatos.

In [ ]:
# Artistas con datos incompletos
query = """
SELECT
    artista_id,
    nombre_artista,
    genero,
    pais_origen
FROM artistas
WHERE pais_origen IS NULL;
"""

artistas_sin_pais = ejecutar_query(query)
print(f"\nArtistas sin país: {len(artistas_sin_pais)}")
artistas_sin_pais

### Ejercicio 2.7: BETWEEN para rangos

Filtrar usuarios registrados en el último año para análisis de nuevos usuarios.

In [ ]:
# Usuarios registrados en 2024
query = """
SELECT
    usuario_id,
    nombre_usuario,
    fecha_registro,
    tipo_suscripcion
FROM usuarios
WHERE fecha_registro BETWEEN '2024-01-01' AND '2024-12-31'
ORDER BY fecha_registro DESC
LIMIT 20;
"""

usuarios_2024 = ejecutar_query(query)
usuarios_2024

---

## Parte 3: Agregación y Métricas

Uso de funciones de agregación para calcular métricas de negocio.

### Ejercicio 3.1: COUNT - Conteo de registros

Calcular métricas básicas de la plataforma.

In [ ]:
# Estadísticas globales de la plataforma
query = """
SELECT
    COUNT(*) AS total_usuarios,
    COUNT(DISTINCT pais) AS paises_activos,
    COUNT(DISTINCT tipo_suscripcion) AS tipos_suscripcion
FROM usuarios;
"""

stats_plataforma = ejecutar_query(query, mostrar_shape=False)
stats_plataforma

### Ejercicio 3.2: COUNT vs COUNT(columna) vs COUNT(DISTINCT)

Comparar diferentes variantes de COUNT para entender datos faltantes.

In [ ]:
# Análisis de completitud de datos en usuarios
query = """
SELECT
    COUNT(*) AS total_registros,
    COUNT(ciudad) AS con_ciudad,
    COUNT(edad) AS con_edad,
    COUNT(DISTINCT pais) AS paises_unicos,
    COUNT(DISTINCT ciudad) AS ciudades_unicas
FROM usuarios;
"""

completitud = ejecutar_query(query, mostrar_shape=False)
print("\nAnálisis de completitud:")
print(completitud)
print(
    f"\nRegistros sin ciudad: {completitud['total_registros'][0] - completitud['con_ciudad'][0]}"
)
print(
    f"Registros sin edad: {completitud['total_registros'][0] - completitud['con_edad'][0]}"
)

### Ejercicio 3.3: SUM y AVG para métricas numéricas

Calcular métricas de engagement: total de reproducciones y promedio por canción.

In [ ]:
# Métricas de reproducciones
query = """
SELECT
    COUNT(*) AS total_eventos_reproduccion,
    SUM(completo) AS reproducciones_completas,
    ROUND(100.0 * SUM(completo) / COUNT(*), 2) AS tasa_completado_pct
FROM reproducciones;
"""

metricas_reprod = ejecutar_query(query, mostrar_shape=False)
metricas_reprod

### Ejercicio 3.4: GROUP BY para análisis por segmentos

Analizar distribución de usuarios por tipo de suscripción.

In [ ]:
# Usuarios por tipo de suscripción
query = """
SELECT
    tipo_suscripcion,
    COUNT(*) AS num_usuarios,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM usuarios), 2) AS porcentaje
FROM usuarios
GROUP BY tipo_suscripcion
ORDER BY num_usuarios DESC;
"""

usuarios_por_tipo = ejecutar_query(query, mostrar_shape=False)
usuarios_por_tipo

In [ ]:
# Visualizar distribución
plt.figure(figsize=(10, 6))
plt.bar(
    usuarios_por_tipo["tipo_suscripcion"],
    usuarios_por_tipo["num_usuarios"],
    color="steelblue",
    edgecolor="black",
)
plt.xlabel("Tipo de Suscripción", fontweight="bold")
plt.ylabel("Número de Usuarios", fontweight="bold")
plt.title("Distribución de Usuarios por Tipo de Suscripción", fontweight="bold")
plt.xticks(rotation=45)
for i, v in enumerate(usuarios_por_tipo["num_usuarios"]):
    plt.text(i, v + 3, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

### Ejercicio 3.5: Múltiples agregaciones por grupo

Calcular estadísticas por género musical.

In [ ]:
# Estadísticas por género musical
query = """
SELECT
    genero,
    COUNT(*) AS num_artistas,
    SUM(seguidores) AS total_seguidores,
    ROUND(AVG(seguidores), 0) AS promedio_seguidores,
    MAX(seguidores) AS max_seguidores
FROM artistas
GROUP BY genero
ORDER BY total_seguidores DESC;
"""

stats_genero = ejecutar_query(query, mostrar_shape=False)
stats_genero

### Ejercicio 3.6: HAVING para filtrar grupos

Identificar países con alta concentración de usuarios (>30 usuarios).

In [ ]:
# Países con más de 30 usuarios
query = """
SELECT
    pais,
    COUNT(*) AS num_usuarios
FROM usuarios
GROUP BY pais
HAVING COUNT(*) > 30
ORDER BY num_usuarios DESC;
"""

paises_grandes = ejecutar_query(query, mostrar_shape=False)
paises_grandes

### Ejercicio 3.7: Combinando WHERE y HAVING

Analizar usuarios Premium por país, solo países con >10 Premium.

In [ ]:
# Concentración de usuarios Premium por país
query = """
SELECT
    pais,
    COUNT(*) AS usuarios_premium
FROM usuarios
WHERE tipo_suscripcion = 'Premium'
GROUP BY pais
HAVING COUNT(*) > 10
ORDER BY usuarios_premium DESC;
"""

premium_por_pais = ejecutar_query(query, mostrar_shape=False)
premium_por_pais

---

## Parte 4: Combinación de Tablas con JOINs

Integrar información de múltiples tablas para análisis complejo.

### Ejercicio 4.1: INNER JOIN básico

Combinar canciones con información de artistas.

In [ ]:
# Canciones con información del artista
query = """
SELECT
    c.cancion_id,
    c.titulo,
    c.duracion_segundos,
    a.nombre_artista,
    a.genero
FROM canciones c
INNER JOIN artistas a ON c.artista_id = a.artista_id
LIMIT 15;
"""

canciones_completas = ejecutar_query(query)
canciones_completas

### Ejercicio 4.2: INNER JOIN con agregación

Contar cuántas canciones tiene cada artista.

In [ ]:
# Artistas con más canciones en catálogo
query = """
SELECT
    a.nombre_artista,
    a.genero,
    COUNT(c.cancion_id) AS num_canciones
FROM artistas a
INNER JOIN canciones c ON a.artista_id = c.artista_id
GROUP BY a.artista_id, a.nombre_artista, a.genero
ORDER BY num_canciones DESC
LIMIT 15;
"""

artistas_productivos = ejecutar_query(query, mostrar_shape=False)
artistas_productivos

### Ejercicio 4.3: LEFT JOIN para incluir registros sin coincidencia

Identificar usuarios que NO han creado playlists.

In [ ]:
# Usuarios sin playlists
query = """
SELECT
    u.usuario_id,
    u.nombre_usuario,
    u.tipo_suscripcion,
    p.playlist_id
FROM usuarios u
LEFT JOIN playlists p ON u.usuario_id = p.usuario_id
WHERE p.playlist_id IS NULL
LIMIT 20;
"""

usuarios_sin_playlists = ejecutar_query(query)
print(f"\nTotal de usuarios sin playlists: {len(usuarios_sin_playlists)}")
usuarios_sin_playlists

### Ejercicio 4.4: Múltiples JOINs

Analizar reproducciones con información completa de usuario, canción y artista.

In [ ]:
# Reproducciones detalladas
query = """
SELECT
    r.fecha_reproduccion,
    u.pais AS pais_usuario,
    u.tipo_suscripcion,
    c.titulo AS cancion,
    a.nombre_artista,
    a.genero,
    r.completo
FROM reproducciones r
INNER JOIN usuarios u ON r.usuario_id = u.usuario_id
INNER JOIN canciones c ON r.cancion_id = c.cancion_id
INNER JOIN artistas a ON c.artista_id = a.artista_id
LIMIT 20;
"""

reproducciones_detalladas = ejecutar_query(query)
reproducciones_detalladas

### Ejercicio 4.5: JOIN con agregación compleja

Top 10 canciones más reproducidas con información del artista.

In [ ]:
# Canciones más populares (por reproducciones recientes)
query = """
SELECT
    c.titulo,
    a.nombre_artista,
    a.genero,
    COUNT(r.reproduccion_id) AS reproducciones_recientes,
    SUM(r.completo) AS reproducciones_completas,
    ROUND(100.0 * SUM(r.completo) / COUNT(r.reproduccion_id), 2) AS tasa_completado_pct
FROM canciones c
INNER JOIN artistas a ON c.artista_id = a.artista_id
INNER JOIN reproducciones r ON c.cancion_id = r.cancion_id
GROUP BY c.cancion_id, c.titulo, a.nombre_artista, a.genero
ORDER BY reproducciones_recientes DESC
LIMIT 10;
"""

top_canciones = ejecutar_query(query, mostrar_shape=False)
top_canciones

---

## Parte 5: Análisis Integrado

Consultas que combinan múltiples conceptos para responder preguntas de negocio.

### Ejercicio 5.1: Análisis RFM simplificado

Identificar usuarios de alto valor basados en frecuencia de uso reciente.

In [ ]:
# Usuarios más activos en últimos 30 días
query = """
SELECT
    u.usuario_id,
    u.nombre_usuario,
    u.pais,
    u.tipo_suscripcion,
    COUNT(r.reproduccion_id) AS reproducciones_30dias,
    SUM(r.completo) AS completas_30dias,
    MAX(r.fecha_reproduccion) AS ultima_reproduccion
FROM usuarios u
INNER JOIN reproducciones r ON u.usuario_id = r.usuario_id
WHERE r.fecha_reproduccion >= DATE('now', '-30 days')
GROUP BY u.usuario_id, u.nombre_usuario, u.pais, u.tipo_suscripcion
HAVING COUNT(r.reproduccion_id) >= 20
ORDER BY reproducciones_30dias DESC
LIMIT 20;
"""

usuarios_activos = ejecutar_query(query, mostrar_shape=False)
usuarios_activos

### Ejercicio 5.2: Análisis de preferencias por país

Géneros musicales más populares por país (top 3 géneros por país).

In [ ]:
# Géneros más escuchados por país
query = """
SELECT
    u.pais,
    a.genero,
    COUNT(r.reproduccion_id) AS reproducciones
FROM reproducciones r
INNER JOIN usuarios u ON r.usuario_id = u.usuario_id
INNER JOIN canciones c ON r.cancion_id = c.cancion_id
INNER JOIN artistas a ON c.artista_id = a.artista_id
WHERE u.pais IN ('España', 'México', 'Argentina', 'Colombia')
GROUP BY u.pais, a.genero
ORDER BY u.pais, reproducciones DESC;
"""

generos_por_pais = ejecutar_query(query, mostrar_shape=False)

# Mostrar top 3 géneros por país
for pais in ["España", "México", "Argentina", "Colombia"]:
    print(f"\n{pais}:")
    top3 = generos_por_pais[generos_por_pais["pais"] == pais].head(3)
    print(top3[["genero", "reproducciones"]])

### Ejercicio 5.3: Análisis de conversión Free a Premium

Comparar comportamiento de usuarios Free vs Premium.

In [ ]:
# Métricas de engagement por tipo de suscripción
query = """
SELECT
    u.tipo_suscripcion,
    COUNT(DISTINCT u.usuario_id) AS usuarios,
    COUNT(r.reproduccion_id) AS total_reproducciones,
    ROUND(COUNT(r.reproduccion_id) * 1.0 / COUNT(DISTINCT u.usuario_id), 2) AS reprod_por_usuario,
    ROUND(100.0 * SUM(r.completo) / COUNT(r.reproduccion_id), 2) AS tasa_completado_pct
FROM usuarios u
LEFT JOIN reproducciones r ON u.usuario_id = r.usuario_id
GROUP BY u.tipo_suscripcion
ORDER BY reprod_por_usuario DESC;
"""

engagement_por_tipo = ejecutar_query(query, mostrar_shape=False)
engagement_por_tipo

### Ejercicio 5.4: Análisis de retención de contenido

Canciones con baja tasa de completado (posibles problemas de calidad).

In [ ]:
# Canciones con baja tasa de completado
query = """
SELECT
    c.titulo,
    a.nombre_artista,
    COUNT(r.reproduccion_id) AS reproducciones,
    SUM(r.completo) AS completas,
    ROUND(100.0 * SUM(r.completo) / COUNT(r.reproduccion_id), 2) AS tasa_completado_pct
FROM canciones c
INNER JOIN artistas a ON c.artista_id = a.artista_id
INNER JOIN reproducciones r ON c.cancion_id = r.cancion_id
GROUP BY c.cancion_id, c.titulo, a.nombre_artista
HAVING COUNT(r.reproduccion_id) >= 10  -- Solo canciones con suficientes datos
ORDER BY tasa_completado_pct ASC
LIMIT 15;
"""

canciones_problematicas = ejecutar_query(query, mostrar_shape=False)
print("\nCanciones con menor tasa de completado:")
canciones_problematicas

### Ejercicio 5.5: Análisis de playlists

Playlists más populares basadas en contenido reproducido.

In [ ]:
# Playlists con más reproducciones agregadas
query = """
SELECT
    p.playlist_id,
    p.nombre_playlist,
    u.nombre_usuario AS creador,
    COUNT(DISTINCT pc.cancion_id) AS num_canciones,
    COUNT(r.reproduccion_id) AS reproducciones_canciones
FROM playlists p
INNER JOIN usuarios u ON p.usuario_id = u.usuario_id
INNER JOIN playlist_canciones pc ON p.playlist_id = pc.playlist_id
LEFT JOIN reproducciones r ON pc.cancion_id = r.cancion_id
GROUP BY p.playlist_id, p.nombre_playlist, u.nombre_usuario
HAVING COUNT(r.reproduccion_id) > 0
ORDER BY reproducciones_canciones DESC
LIMIT 15;
"""

playlists_populares = ejecutar_query(query, mostrar_shape=False)
playlists_populares

---

## Conclusiones del Laboratorio

En este laboratorio se practicaron las siguientes competencias SQL:

### 1. Consultas Básicas
- SELECT con columnas específicas vs SELECT *
- LIMIT para muestras
- Exploración de esquemas con PRAGMA

### 2. Filtrado y Ordenamiento
- WHERE con operadores de comparación
- AND, OR, NOT para condiciones complejas
- IN, BETWEEN, LIKE para patrones
- IS NULL para valores faltantes
- ORDER BY para ordenar resultados

### 3. Agregaciones
- COUNT(*), COUNT(columna), COUNT(DISTINCT)
- SUM, AVG, MAX, MIN
- GROUP BY para análisis por segmentos
- HAVING para filtrar grupos

### 4. JOINs
- INNER JOIN para coincidencias
- LEFT JOIN para incluir registros sin match
- Múltiples JOINs para integrar varias tablas
- JOINs con agregaciones

### 5. Análisis Integrado
- Combinación de WHERE, GROUP BY, HAVING, JOINs
- Cálculo de métricas de negocio (engagement, retención, conversión)
- Análisis RFM simplificado
- Segmentación por demografía y comportamiento

### Aplicaciones en Industria

Las consultas practicadas en este laboratorio son representativas de análisis reales en:

- **Plataformas de streaming**: Análisis de engagement, curación de contenido
- **E-commerce**: Segmentación de clientes, análisis de productos
- **SaaS**: Métricas de activación, retención, conversión
- **Marketing digital**: Análisis de campañas, atribución
- **Fintech**: Análisis transaccional, detección de patrones

La capacidad de escribir SQL eficiente es fundamental para extraer y preparar datos que alimentan pipelines de machine learning, dashboards y reportes automatizados.

In [ ]:
# Cerrar conexión a la base de datos
conn.close()
print("✓ Conexión cerrada")